1. Ручное глобальное выравнивание (1 балл)
Вручную постройте матрицу Нидлмана-Вунша для строк X: ATGCCG и Y : ATCG.
Параметры: Match = +3, Mismatch = −3, Gap = −2.
Укажите итоговый Score, матрицу и само выравнивание

In [2]:
x = 'ATGCCG'
y = 'ATCG'
match = 3
mismatch = -3
gap = -2
n = len(x)
m = len(y)


In [4]:
import numpy as np

In [5]:
matrix = np.zeros((n+1, m+1))

In [6]:
matrix[0, :] = np.arange(0, m+1)*gap
matrix[:, 0] = np.arange(0, n+1)*gap

In [7]:
for i in range(1, n + 1):
    for j in range(1, m + 1):
        score = match if x[i-1] == y[j-1] else mismatch
        
        matrix[i, j] = max(
            matrix[i-1, j-1] + score,  
            matrix[i-1, j] + gap,      
            matrix[i, j-1] + gap       
        )

In [20]:
matrix

array([[  0.,  -2.,  -4.,  -6.,  -8.],
       [ -2.,   3.,   1.,  -1.,  -3.],
       [ -4.,   1.,   6.,   4.,   2.],
       [ -6.,  -1.,   4.,   3.,   7.],
       [ -8.,  -3.,   2.,   7.,   5.],
       [-10.,  -5.,   0.,   5.,   4.],
       [-12.,  -7.,  -2.,   3.,   8.]])

In [8]:
s1 = '' 
s2 = ''
i, j = n, m

while i > 0 or j > 0:
    current_score = matrix[i][j]

    if i > 0 and j > 0 and matrix[i][j] == matrix[i-1][j-1] + (match if x[i-1] == y[j-1] else mismatch):
        s1 += x[i-1]
        s2 += y[j-1]
        i -= 1
        j -= 1
        
    elif i > 0 and matrix[i][j] == matrix[i-1][j] + gap:
        s1 += x[i-1]
        s2 += '-'
        i -= 1
        
    elif j > 0 and matrix[i][j] == matrix[i][j-1] + gap:
        s1 += '-'
        s2 += y[j-1]
        j -= 1

alignment1 = s1[::-1]
alignment2 = s2[::-1]

In [9]:
alignment1

'ATGCCG'

In [10]:
alignment2

'AT--CG'

In [ ]:
matrix[-1, -1] #score

np.float64(8.0)

2. Ручное Локальное выравнивание (1 балл)
Для тех же строк и параметров постройте матрицу Смита-Ватермана.
Укажите итоговый Score, матрицу и само выравнивание

In [21]:
sw_matrix = np.zeros((n+1, m+1))

In [22]:
for i in range(1, n + 1):
    for j in range(1, m + 1):
        score = match if x[i-1] == y[j-1] else mismatch
        
        sw_matrix[i, j] = max(0,
            sw_matrix[i-1, j-1] + score,  
            sw_matrix[i-1, j] + gap,      
            sw_matrix[i, j-1] + gap       
        )

In [23]:
sw_matrix

array([[0., 0., 0., 0., 0.],
       [0., 3., 1., 0., 0.],
       [0., 1., 6., 4., 2.],
       [0., 0., 4., 3., 7.],
       [0., 0., 2., 7., 5.],
       [0., 0., 0., 5., 4.],
       [0., 0., 0., 3., 8.]])

In [24]:
i, j = np.unravel_index(np.argmax(matrix), matrix.shape)

s1 = '' 
s2 = ''

while i > 0 and j > 0 and matrix[i][j] > 0:
    current_score = matrix[i][j]

    if matrix[i][j] == matrix[i-1][j-1] + (match if x[i-1] == y[j-1] else mismatch):
        s1 += x[i-1]
        s2 += y[j-1]
        i -= 1
        j -= 1

    elif matrix[i][j] == matrix[i-1][j] + gap:
        s1 += x[i-1]
        s2 += '-'
        i -= 1

    elif matrix[i][j] == matrix[i][j-1] + gap:
        s1 += '-'
        s2 += y[j-1]
        j -= 1

alignment1 = s1[::-1]
alignment2 = s2[::-1]

In [25]:
alignment1

'ATGCCG'

In [26]:
alignment2

'AT--CG'

In [27]:
sw_matrix

array([[0., 0., 0., 0., 0.],
       [0., 3., 1., 0., 0.],
       [0., 1., 6., 4., 2.],
       [0., 0., 4., 3., 7.],
       [0., 0., 2., 7., 5.],
       [0., 0., 0., 5., 4.],
       [0., 0., 0., 3., 8.]])

3. Аффинные штрафы за гэпы (1 балл)
Реализуйте алгоритм Нидлмана-Вунша на python и рассчитайте Score для выравнивания ATGCAGCAGCAGCCA
и ATATAT при двух моделях:
• Линейный штраф: Gap = −4
• Аффинный штраф: Открытие гэпа (Open) = −10, продолжение гэпа (Extend) = −1.
Объясните, какую биологическую особенность лучше описывает аффинная модель. Приложите матрицы
и выравнивания.

In [28]:
x = 'ATGCAGCAGCAGCCA'
y = 'ATATAT'
gap = -4
open_gap = -10
extend_gap = -1

In [ ]:
M = np.zeros((n + 1, m + 1))
Ix = np.full((n + 1, m + 1), -np.inf)
Iy = np.full((n + 1, m + 1), -np.inf)

for i in range(1, n + 1):
    for j in range(1, m + 1):
        score = match if x[i-1] == y[j-1] else mismatch
        
        M[i, j] = score + max(M[i-1, j-1], Ix[i-1, j-1], Iy[i-1, j-1])
        
        Ix[i, j] = max(
            M[i-1, j] + open_gap,    
            Ix[i-1, j] + extend_gap  
        )
        
        Iy[i, j] = max(
            M[i, j-1] + open_gap, 
            Iy[i, j-1] + extend_gap
        )


In [ ]:
align_x = ""
align_y = ""

i, j = n, m

scores = [M[i, j], Ix[i, j], Iy[i, j]]
state = scores.index(max(scores)) 

while i > 0 or j > 0:
    if state == 0:
        align_x = x[i-1] + align_x
        align_y = y[j-1] + align_y
  
        prev_scores = [M[i-1, j-1], Ix[i-1, j-1], Iy[i-1, j-1]]
        state = prev_scores.index(max(prev_scores))
        i -= 1
        j -= 1

    # СОСТОЯНИЕ Ix (Вставка в X / Гэп в Y) -> Идем ВВЕРХ
    elif state == 1:
        align_x = x[i-1] + align_x
        align_y = "-" + align_y
        # Мы либо продлили гэп (остались в Ix), либо открыли его (ушли в M)
        if Ix[i, j] == Ix[i-1, j] + extend_gap:
            state = 1 # остаемся в Ix
        else:
            state = 0 # прыгаем в M
        i -= 1

    # СОСТОЯНИЕ Iy (Делеция в X / Гэп в X) -> Идем ВЛЕВО
    elif state == 2:
        align_x = "-" + align_x
        align_y = y[j-1] + align_y
        # Мы либо продлили гэп (остались в Iy), либо открыли его (ушли в M)
        if Iy[i, j] == Iy[i, j-1] + extend_gap:
            state = 2 # остаемся в Iy
        else:
            state = 0 # прыгаем в M
        j -= 1

In [35]:
M

array([[  0.,   0.,   0.,   0.,   0.],
       [  0.,   3.,  -3.,   3.,  -3.],
       [  0.,  -3.,   6.,  -6.,   6.],
       [  0.,  -3.,  -6.,   3.,  -7.],
       [  0.,  -3.,  -6.,  -7.,   0.],
       [  0.,   3.,  -6.,  -2., -10.],
       [  0.,  -3.,   0.,  -9.,  -5.]])

In [37]:
Ix

array([[-inf, -inf, -inf, -inf, -inf],
       [-inf, -10., -10., -10., -10.],
       [-inf,  -7., -11.,  -7., -11.],
       [-inf,  -8.,  -4.,  -8.,  -4.],
       [-inf,  -9.,  -5.,  -7.,  -5.],
       [-inf, -10.,  -6.,  -8.,  -6.],
       [-inf,  -7.,  -7.,  -9.,  -7.]])

In [38]:
Iy

array([[-inf, -inf, -inf, -inf, -inf],
       [-inf, -10.,  -7.,  -8.,  -7.],
       [-inf, -10., -11.,  -4.,  -5.],
       [-inf, -10., -11., -12.,  -7.],
       [-inf, -10., -11., -12., -13.],
       [-inf, -10.,  -7.,  -8.,  -9.],
       [-inf, -10., -11., -10., -11.]])

In [33]:
align_x

'ATGCAG'

In [34]:
align_y

'AT--AT'

дефолтный алгоритм может насовать много единичных гепов, что ирл, конечно, происходит очень редко. данный алгоритм вводит как бы потенциальный барьер для введения гепа, после которого продолжение гепа(типа у нас крупная делеция произошла) сделать дёшево.